# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row represents exactly **one unique content page** (`content_id`) belonging to a single publisher (`client_id`).

**Time Window:** The performance metrics represent a 90-day historical observation window (`impressions_90d`), evaluated against prior periods to determine the current decay trend.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify unit of analysis: content_id should be completely unique
is_unique = df['content_id'].is_unique
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")
print(f"Is the grain strictly 1 row = 1 content piece? {is_unique}")
assert is_unique, "Grain violation: Duplicate content_ids detected!"

Total rows: 30,000
Unique content_ids: 30,000
Is the grain strictly 1 row = 1 content piece? True


## 2. Fields: feature / label / context / excluded

*   **Features ($X$):** `days_since_last_update`, `content_age_days`, `impressions_90d`, `avg_position`, `ctr`. (These are pre-outcome signals reflecting freshness and visibility).
*   **Label ($y$):** `is_declining_label` (Derived mathematically as $1$ if `trend_direction` == "down", otherwise $0$).
*   **Context:** `content_id`, `client_id`, `content_type`. (Required for indexing and grouped cross-validation routing).
*   **Excluded:**
    *   `trend_pct`: Excluded due to target leakage (it is the exact percentage change used to calculate the label).
    *   `search_volume`: Excluded due to empirical irrelevance (prior EDA showed $r \approx 0.001$ with actual traffic).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Construct the explicit schema vectors
features = ['days_since_last_update', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr']
context = ['content_id', 'client_id', 'content_type']
excluded = ['trend_pct', 'search_volume']

# Instantiate the label
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
target = 'is_declining_label'

print("Contract Schema Initialized.")
print(f"Feature space dimensionality: {len(features)}")

Contract Schema Initialized.
Feature space dimensionality: 5


## 3. Verify it with queries (grain, counts, missing values, windows)

We must audit our selected feature matrix for nulls to determine if imputation or dropping is required, and establish the base rate of our target label to ensure balanced evaluation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Check for missing values in our active feature space
print("Missing Value Count per Feature:")
print(df[features].isna().sum())

# 2. Verify Label Base Rate
base_rate = df[target].mean()
print(f"\nTarget Base Rate (Declining Pages): {base_rate:.3f}")
print(f"Majority Class Base Rate: {max(base_rate, 1 - base_rate):.3f}")

# 3. Verify Client Unbalance (Context check)
client_counts = df['client_id'].value_counts()
print(f"\nClient Distribution:\nMax pages per client: {client_counts.max()}\nMin pages per client: {client_counts.min()}")

Missing Value Count per Feature:
days_since_last_update    0
content_age_days          0
impressions_90d           0
avg_position              0
ctr                       0
dtype: int64

Target Base Rate (Declining Pages): 0.542
Majority Class Base Rate: 0.542

Client Distribution:
Max pages per client: 7008
Min pages per client: 3


## 4. Data limits

*   **Unobservable Algorithmic Shifts:** This dataset captures the symptoms of traffic decay, but it cannot observe the cause (e.g., proprietary Google Core Updates). We are modeling correlation, not causality.
*   **Feature Sparsity:** Features like `word_count` suffer from ~25% missing values. We excluded it from the core feature contract to maintain matrix density, relying instead on high-fidelity Google Search Console telemetry.
*   **Unbalanced Panel:** The dataset exhibits heavy client skew (some clients have thousands of rows, others very few). We must use a strictly grouped cross-validation strategy to prevent evaluation bias.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify the word_count sparsity mentioned in limits
word_count_missing = df['word_count'].isna().mean()
print(f"word_count missing rate: {word_count_missing:.1%}")
print("Confirmation: Exclusion of word_count preserves 100% of the dataset for the core features.")

word_count missing rate: 25.7%
Confirmation: Exclusion of word_count preserves 100% of the dataset for the core features.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.